# 🛡️ **Data Preprocessing & Anonimisasi**
Notebook ini berisi implementasi 6 teknik anonimisasi data (Masking, Pseudonymization, Generalization, Noise Addition, Encryption, dan Data Shuffling) untuk mengamankan data demografis dan sosial-ekonomi siswa sebelum dimasukkan ke dalam model *Machine Learning*.

In [18]:
import pandas as pd
import numpy as np
import hashlib
from IPython.display import display
from cryptography.fernet import Fernet


In [4]:
np.random.seed(42)

df = pd.read_csv("data_dummy.csv")

print("=" * 100)
print("  DATA ASLI (sebelum anonimisasi)")
print("=" * 100)
print(df[["NISN", "Nama_Lengkap", "Rata_Nilai", "Kehadiran_Persen",
        "Penghasilan_Keluarga_Bulan", "Usia"]].head(5).to_string())
print("-" * 100)

  DATA ASLI (sebelum anonimisasi)
         NISN      Nama_Lengkap  Rata_Nilai  Kehadiran_Persen  Penghasilan_Keluarga_Bulan  Usia
0  3746317213      Doni Pratama          83                93                     2700000    18
1  9697354961     Salsa Pratama          67                83                     2100000    14
2  2181241943      Andi Nugroho          69                79                     3900000    13
3  1958682846    Reni Kurniawan          66                84                     2300000    13
4  4163119785  Indah Situmorang          76                81                     1900000    17
----------------------------------------------------------------------------------------------------


## 👤 **Data Masking**
- **Teknik:** Menyembunyikan sebagian karakter dengan simbol bintang (*).
- **Target:** `Nama_Lengkap`
- **Tujuan:** Mencegah identifikasi langsung tanpa kehilangan format nama.

In [20]:
def masking_nama(nama: str) -> str:
    # Pertahankan 3 huruf pertama, bintangi sisanya per kata.
    kata = nama.split()
    hasil = []
    for i, k in enumerate(kata):
        if i == 0:
            # Kata pertama: tampilin 3 huruf awal
            hasil.append(k[:3] + "*" * max(0, len(k) - 3))
        else:
            # Kata berikutnya: sembunyikan semua kecuali huruf kapital awal
            hasil.append(k[0] + "*" * (len(k) - 1))
    return " ".join(hasil)

df["Nama_Masking"] = df["Nama_Lengkap"].apply(masking_nama)

print("=" * 30)
print("DATA MASKING — Nama_Lengkap")
print("=" * 30)
print(df[["Nama_Lengkap", "Nama_Masking"]].head(4).to_string(index=False))

DATA MASKING — Nama_Lengkap
  Nama_Lengkap   Nama_Masking
  Doni Pratama   Don* P******
 Salsa Pratama  Sal** P******
  Andi Nugroho   And* N******
Reni Kurniawan Ren* K********


## 🔐 **Anggota 2: Pseudonymization (Hashing Searah)**
- **Teknik:** Mengganti identifier asli dengan token acak yang konsisten.
- **Target:** `NISN`
- **Tujuan:** NISN tidak bisa dibaca langsung, tapi satu siswa tetap punya satu ID unik (konsisten lintas tabel/join).

In [23]:
def hash_nisn(nisn: str) -> str:
    # SHA-256, ambil 16 karakter pertama sebagai pseudo-ID.
    return hashlib.sha256(nisn.encode()).hexdigest()[:16]

df["NISN_Hash"] = df["NISN"].astype(str).apply(hash_nisn)

print("=" * 30)
print("PSEUDONYMIZATION — NISN")
print("=" * 30)
print(df[["NISN", "NISN_Hash"]].head(4).to_string(index=False))

PSEUDONYMIZATION — NISN
      NISN        NISN_Hash
3746317213 7dcf6a4fefc70dbe
9697354961 e27f18e29084e369
2181241943 fb9f4d84cd4877e5
1958682846 01693fbd7712e25f


## 📊 **Generalization (Pengelompokan Rentang)**
- **Teknik:** Mengganti nilai presisi tinggi dengan label rentang umum.
- **Target:** `Penghasilan_Keluarga_Bulan`, `Usia`, `Jarak_Ke_Sekolah_KM`
- **Tujuan:** Mengurangi granularitas data sensitif (angka pasti → bracket).

In [17]:
# Generalisasi penghasilan keluarga
batas_gaji   = [0, 1_000_000, 2_500_000, 5_000_000, float("inf")]
label_gaji   = ["< 1 Juta", "1 - 2.5 Juta", "2.5 - 5 Juta", "> 5 Juta"]
df["Penghasilan_Generalisasi"] = pd.cut(
    df["Penghasilan_Keluarga_Bulan"], bins=batas_gaji, labels=label_gaji
)

# Generalisasi usia → kelompok umur
batas_usia   = [0, 13, 15, 18, float("inf")]
label_usia   = ["≤ 13 th", "14–15 th", "16–18 th", "> 18 th"]
df["Usia_Generalisasi"] = pd.cut(
    df["Usia"], bins=batas_usia, labels=label_usia
)

# Generalisasi jarak ke sekolah
batas_jarak  = [0, 2, 5, 10, float("inf")]
label_jarak  = ["Dekat (≤2 km)", "Sedang (2–5 km)", "Jauh (5–10 km)", "Sangat Jauh (>10 km)"]
df["Jarak_Generalisasi"] = pd.cut(
    df["Jarak_Ke_Sekolah_KM"], bins=batas_jarak, labels=label_jarak
)

print("=" * 120)
print("GENERALIZATION — Penghasilan, Usia, Jarak")
print("=" * 120)
print(df[["Penghasilan_Keluarga_Bulan", "Penghasilan_Generalisasi",
          "Usia", "Usia_Generalisasi",
          "Jarak_Ke_Sekolah_KM", "Jarak_Generalisasi"]].head(4).to_string(index=False))

GENERALIZATION — Penghasilan, Usia, Jarak
 Penghasilan_Keluarga_Bulan Penghasilan_Generalisasi  Usia Usia_Generalisasi  Jarak_Ke_Sekolah_KM Jarak_Generalisasi
                    2700000             2.5 - 5 Juta    18          16–18 th                  0.4      Dekat (≤2 km)
                    2100000             1 - 2.5 Juta    14          14–15 th                  4.4    Sedang (2–5 km)
                    3900000             2.5 - 5 Juta    13           ≤ 13 th                  3.5    Sedang (2–5 km)
                    2300000             1 - 2.5 Juta    13           ≤ 13 th                  1.7      Dekat (≤2 km)


## 📈 **Noise Addition (Penambahan Gangguan Acak)**
- **Teknik:** Menambahkan nilai desimal acak (distribusi normal) ke data numerik.
- **Target:** `Rata_Nilai`, `Kehadiran_Persen`
- **Tujuan:** Nilai tidak lagi presisi 100%, tapi distribusi statistiknya (mean, std) masih cukup akurat untuk keperluan analitik.

In [27]:
# Noise pada nilai akademik — standar deviasi 2 poin
noise_nilai      = np.random.normal(loc=0, scale=2.0, size=len(df))
df["Nilai_Noise"] = np.clip(
    np.round(df["Rata_Nilai"] + noise_nilai, 1), 0, 100
)

# Noise pada kehadiran — standar deviasi 1.5 poin
noise_kehadiran      = np.random.normal(loc=0, scale=1.5, size=len(df))
df["Kehadiran_Noise"] = np.clip(
    np.round(df["Kehadiran_Persen"] + noise_kehadiran, 1), 0, 100
)

print("=" * 60)
print("NOISE ADDITION — Rata_Nilai & Kehadiran_Persen")
print("=" * 60)
print(df[["Rata_Nilai", "Nilai_Noise",
          "Kehadiran_Persen", "Kehadiran_Noise"]].head(4).to_string(index=False))

NOISE ADDITION — Rata_Nilai & Kehadiran_Persen
 Rata_Nilai  Nilai_Noise  Kehadiran_Persen  Kehadiran_Noise
         83         79.2                93             94.8
         67         65.3                83             82.1
         69         68.2                79             79.1
         66         69.8                84             84.1


## 🔑 **Encryption (Enkripsi Simetris Dua Arah)**
- **Teknik:** Mengenkripsi data sensitif dengan kunci rahasia (Fernet/AES).
- **Target:** `Nama_Lengkap`
- **Tujuan:** Data bisa dibuka kembali (decrypt) oleh Super Admin yang memegang kunci.

In [39]:
kunci_rahasia = Fernet.generate_key()
cipher        = Fernet(kunci_rahasia)

df["Nama_Enkripsi"] = df["Nama_Lengkap"].apply(
    lambda x: cipher.encrypt(x.encode()).decode()
)

# Verifikasi: decrypt balik untuk membuktikan reversible
nama_terdekripsi = cipher.decrypt(df["Nama_Enkripsi"].iloc[0].encode()).decode()

print("\nENCRYPTION — Nama_Lengkap")
print(f"  Asli       : {df['Nama_Lengkap'].iloc[0]}")
print(f"  Terenkripsi: {df['Nama_Enkripsi'].iloc[0][:50]}...")
print(f"  Terdekripsi: {nama_terdekripsi} (reversible dengan kunci)")
print(f"\n ⚠️ Kunci Rahasia (simpan terpisah, JANGAN disertakan di CSV):")
print(f"  {kunci_rahasia.decode()}")


ENCRYPTION — Nama_Lengkap
  Asli       : Doni Pratama
  Terenkripsi: gAAAAABp3EJk2g7e7-48EFJFUX8-b_OaZLtG58GP4haQ0SHcAJ...
  Terdekripsi: Doni Pratama (reversible dengan kunci)

 ⚠️ Kunci Rahasia (simpan terpisah, JANGAN disertakan di CSV):
  x9Ag1ruAfkzwtqhUrWeHxHa53da2pN75zqFG22GhqFY=


## 🔀 **Data Shuffling (Pengacakan Vertikal)**
- **Teknik:** Mengacak urutan nilai dalam satu kolom sehingga tidak lagi cocok dengan baris/individu yang seharusnya.
- **Target:** `Pekerjaan_Ayah`
- **Tujuan:** Profil pekerjaan tidak bisa dikaitkan langsung ke siswa tertentu, tapi distribusi pekerjaan secara keseluruhan tetap sama.

In [43]:
df["Pekerjaan_Ayah_Shuffled"] = (
    df["Pekerjaan_Ayah"]
    .sample(frac=1, random_state=99)
    .reset_index(drop=True)
)

print("\nDATA SHUFFLING — Pekerjaan_Ayah")
print(df[["Nama_Lengkap", "Pekerjaan_Ayah", "Pekerjaan_Ayah_Shuffled"]].head(6).to_string(index=False))
print("\n  Distribusi sebelum vs sesudah (harus sama):")
dist_before = df["Pekerjaan_Ayah"].value_counts()
dist_after  = df["Pekerjaan_Ayah_Shuffled"].value_counts()
cmp = pd.DataFrame({"Sebelum": dist_before, "Sesudah": dist_after})
print(cmp.to_string())


DATA SHUFFLING — Pekerjaan_Ayah
    Nama_Lengkap  Pekerjaan_Ayah Pekerjaan_Ayah_Shuffled
    Doni Pratama          Petani                Pedagang
   Salsa Pratama        Pedagang                Pedagang
    Andi Nugroho    Buruh Harian            Buruh Harian
  Reni Kurniawan    Buruh Harian         Karyawan Swasta
Indah Situmorang Karyawan Swasta           Tidak Bekerja
     Rizky Lubis          Petani                Pedagang

  Distribusi sebelum vs sesudah (harus sama):
                 Sebelum  Sesudah
Buruh Harian         107      107
Karyawan Swasta       70       70
Meninggal              9        9
PNS/TNI/Polri         34       34
Pedagang              88       88
Petani                88       88
Tidak Bekerja         52       52
Wiraswasta            52       52


## **Finishing: Penggabungan Dataset Final**
Memfilter dan menggabungkan hanya kolom yang sudah melalui proses anonimisasi untuk diekspor menjadi *dataset* yang siap dilatih oleh model *Machine Learning*.

In [46]:
df_aman = pd.DataFrame({
    # Identitas anonim
    "ID_Siswa"              : df["NISN_Hash"],                  # Anggota 2
    "Nama_Siswa"            : df["Nama_Masking"],               # Anggota 1

    # Demografis — generalisasi
    "Kelompok_Usia"         : df["Usia_Generalisasi"],          # Anggota 3
    "Kelas"                 : df["Kelas"],
    "Jenis_Kelamin"         : df["Jenis_Kelamin"],
    "Jurusan"               : df["Jurusan"],

    # Akademik — dengan noise
    "Nilai_Akhir"           : df["Nilai_Noise"],                # Anggota 4
    "Kehadiran_Persen"      : df["Kehadiran_Noise"],            # Anggota 4
    "Jumlah_Remedial"       : df["Jumlah_Remedial"],
    "Pernah_Tinggal_Kelas"  : df["Pernah_Tinggal_Kelas"],
    "Siswa_Bekerja"         : df["Siswa_Bekerja"],

    # Keluarga
    "Status_Ortu"           : df["Status_Ortu"],
    "Jumlah_Saudara"        : df["Jumlah_Saudara"],
    "Pendidikan_Ortu"       : df["Pendidikan_Tertinggi_Ortu"],
    "Pekerjaan_Ayah"        : df["Pekerjaan_Ayah_Shuffled"],    # Anggota 6

    # Ekonomi — generalisasi
    "Ekonomi_Keluarga"      : df["Penghasilan_Generalisasi"],   # Anggota 3
    "Jarak_Ke_Sekolah"      : df["Jarak_Generalisasi"],         # Anggota 3
    "Kondisi_Rumah"         : df["Kondisi_Rumah"],
    "Akses_Internet"        : df["Akses_Internet"],

    # Bantuan sosial
    "Penerima_KIP"          : df["Penerima_KIP"],
    "Penerima_PKH"          : df["Penerima_PKH"],
    "Terdaftar_DTKS"        : df["Terdaftar_DTKS"],

    # Target label ML
    "Status_Risiko"         : df["Status_Risiko"],
    "Kelayakan_Bansos"      : df["Kelayakan_Bansos"],
})

df_aman.to_csv("data_siswa_anonimisasi.csv", index=False)

print("\n" + "=" * 65)
print("  DATASET FINAL — SIAP MOBIL LEJEN AS ML")
print("=" * 65)
print(f"  Baris    : {len(df_aman)}")
print(f"  Kolom    : {len(df_aman.columns)}")
print(f"\n  Kolom yang DIHAPUS dari dataset final:")
print(f"    ✗ NISN (asli)         → diganti NISN_Hash")
print(f"    ✗ Nama_Lengkap (asli) → diganti Nama_Masking")
print(f"    ✗ Penghasilan (angka) → diganti bracket kategori")
print(f"    ✗ Usia (angka pasti)  → diganti kelompok umur")
print(f"    ✗ Jarak (angka pasti) → diganti kategori jarak")
print(f"\nPreview dataset akhir:")
print(df_aman.head(3).T.to_string())
print("\n✅ File 'data_siswa_anonimisasi.csv' berhasil disimpan!")


  DATASET FINAL — SIAP MOBIL LEJEN AS ML
  Baris    : 500
  Kolom    : 24

  Kolom yang DIHAPUS dari dataset final:
    ✗ NISN (asli)         → diganti NISN_Hash
    ✗ Nama_Lengkap (asli) → diganti Nama_Masking
    ✗ Penghasilan (angka) → diganti bracket kategori
    ✗ Usia (angka pasti)  → diganti kelompok umur
    ✗ Jarak (angka pasti) → diganti kategori jarak

Preview dataset akhir:
                                     0                 1                 2
ID_Siswa              7dcf6a4fefc70dbe  e27f18e29084e369  fb9f4d84cd4877e5
Nama_Siswa                Don* P******     Sal** P******      And* N******
Kelompok_Usia                 16–18 th          14–15 th           ≤ 13 th
Kelas                               12                 7                 7
Jenis_Kelamin                Laki-laki         Perempuan         Perempuan
Jurusan                       Kejuruan                 -                 -
Nilai_Akhir                       79.2              65.3              68.2
Kehadiran_